# Do pruning and quantization compose? — main study (ESANN 2027)
Single self-contained Colab notebook. Runtime → Change runtime type → **T4 GPU** (free) is enough.

* `MODE = "pilot"`: SmolLM2-135M, 6 configurations, small calibration; ends with a PASS/FAIL validation report.
* `MODE = "full"`: the 93 runs of the paper (3 models × 25 configurations + 18 seed repeats on SmolLM2-135M).

Results are appended to `MyDrive/esann2027/results_<MODE>.jsonl` after every configuration; re-running skips finished ones (resume-safe).

In [ ]:
# Cell 1 — install + environment
!pip install -q -U lm-eval
import torch, transformers, lm_eval, datasets, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| lm_eval", getattr(lm_eval, "__version__", "?"), "| datasets", datasets.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0),
      "| memory GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# Cell 2 — configuration
MODE = "full"             # pilot passed on 135M; "full" runs the whole study
SAVE_TO_DRIVE = True      # keep results across Colab sessions
SEED = 0
SEQLEN = 2048
GROUPSIZE = 128           # GPTQ group size, as in the IJCNN 2025 paper
TASKS = ["arc_easy", "piqa", "hellaswag"]
MAIN_METRIC = {"arc_easy": "acc,none", "piqa": "acc,none", "hellaswag": "acc_norm,none"}
CHANCE = {"arc_easy": 0.25, "piqa": 0.5, "hellaswag": 0.25}

if MODE == "pilot":
    MODELS = ["HuggingFaceTB/SmolLM2-135M"]
    N_CALIB, PPL_WINDOWS, EVAL_LIMIT = 16, 20, 100
    GRID = [("dense", 0.0, 16, 0), ("prune", 0.25, 16, 0), ("quant", 0.0, 4, 0),
            ("joint", 0.25, 4, 0), ("p_then_q", 0.25, 4, 0), ("q_then_p", 0.25, 4, 0)]
    EXTRA_SEEDS = {}
else:
    # run order: smallest model first; results are saved after every config, so a disconnect loses nothing
    MODELS = ["HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-360M", "Qwen/Qwen2.5-0.5B"]
    N_CALIB, PPL_WINDOWS, EVAL_LIMIT = 128, None, 500
    SPARS, BITS = [0.25, 1/3, 0.5], [4, 3]
    GRID = [("dense", 0.0, 16, 0)] + [("prune", s, 16, 0) for s in SPARS] + [("quant", 0.0, b, 0) for b in [8, 4, 3]] \
         + [(o, s, b, 0) for s in SPARS for b in BITS for o in ("joint", "p_then_q", "q_then_p")]
    # seed robustness (different calibration samples) on the smallest model, two key cells, all orders
    EXTRA_SEEDS = {"HuggingFaceTB/SmolLM2-135M": [(o, s, 4, sd) for sd in (1, 2) for s in (0.25, 0.5)
                                                   for o in ("prune", "joint", "p_then_q", "q_then_p")]
                                                 + [("quant", 0.0, 4, sd) for sd in (1, 2)]}
n_total = sum(len(GRID) + len(EXTRA_SEEDS.get(m, [])) for m in MODELS)
print(f"MODE={MODE} | models={MODELS} | total configs={n_total} | calib={N_CALIB}x{SEQLEN} | eval limit={EVAL_LIMIT}")

In [ ]:
# Cell 3 — core: OBS compression (SparseGPT/GPTQ written from scratch), calibration, perplexity
# ===================== CORE: calibration, OBS compression, evaluation =====================
import math, time, json, gc, random
import torch, torch.nn as nn

def tcr(sparsity, bits):
    """Theoretical compression rate (IJCNN 2025 definition), sparsity in [0,1], 16-bit baseline."""
    return 1.0 - (bits / 16.0) * (1.0 - sparsity)

# ---------- quantizer (asymmetric min-max, per output row, per group of input columns) ----------
def find_params(x, bits):
    maxq = 2 ** bits - 1
    xmin = torch.minimum(x.min(1).values, torch.zeros(x.shape[0], device=x.device))
    xmax = torch.maximum(x.max(1).values, torch.zeros(x.shape[0], device=x.device))
    both0 = (xmin == 0) & (xmax == 0)
    xmin[both0], xmax[both0] = -1.0, 1.0
    scale = (xmax - xmin) / maxq
    zero = torch.round(-xmin / scale)
    return scale, zero, maxq

def quantize(x, scale, zero, maxq):
    q = torch.clamp(torch.round(x / scale) + zero, 0, maxq)
    return scale * (q - zero)

# ---------- unified OBS sweep (SparseGPT/GPTQ family) ----------
@torch.no_grad()
def obs_sweep(W, H, sparsity=0.0, bits=16, fixed_mask=None, fixed_grid=None,
              blocksize=128, groupsize=128, percdamp=0.01):
    """
    One column-wise OBS pass over W (rows x cols) with Hessian H (cols x cols).
      sparsity>0 and fixed_mask None -> select an unstructured mask (SparseGPT, adaptive per block)
      fixed_mask given               -> keep exactly these weights at zero (mask preserved)
      bits<16                        -> quantize every column (GPTQ); grid per (row, group)
      fixed_grid given               -> re-use an existing grid {group_idx: (scale, zero)} instead of fitting one
    Returns (W_hat, mask, grid).
    """
    W = W.clone().float()
    rows, cols = W.shape
    H = H.clone()
    bad = int((~torch.isfinite(H)).sum().item())
    if bad:
        print(f"    WARNING: {bad} non-finite Hessian entries (fp16 overflow in calibration) -> set to 0")
        H = torch.nan_to_num(H, nan=0.0, posinf=0.0, neginf=0.0)
    dead = torch.diag(H) == 0
    H[dead, dead] = 1
    W[:, dead] = 0
    damp = percdamp * torch.mean(torch.diag(H))
    idx = torch.arange(cols, device=W.device)
    Hinv = None
    for attempt in range(5):
        try:
            Hd = H.clone(); Hd[idx, idx] += damp * (10 ** attempt)
            L = torch.linalg.cholesky(Hd)
            Hinv = torch.cholesky_inverse(L)
            Hinv = torch.linalg.cholesky(Hinv, upper=True)
            break
        except Exception:
            Hinv = None
    if Hinv is None:
        print("    WARNING: Cholesky failed after 5 damping levels -> diagonal Hessian fallback")
        dg = torch.diag(H).clamp_min(0)
        Hinv = torch.diag(1.0 / torch.sqrt(dg + percdamp * dg.mean() + 1e-8))
    quant = bits < 16
    maxq = 2 ** bits - 1
    mask = torch.zeros_like(W, dtype=torch.bool)
    grid = {}
    for i1 in range(0, cols, blocksize):
        i2 = min(i1 + blocksize, cols); count = i2 - i1
        W1 = W[:, i1:i2].clone()
        Q1 = torch.zeros_like(W1); Err1 = torch.zeros_like(W1)
        Hinv1 = Hinv[i1:i2, i1:i2]
        if fixed_mask is not None:
            mask1 = fixed_mask[:, i1:i2].clone()
        elif sparsity > 0:
            tmp = W1 ** 2 / (torch.diag(Hinv1).reshape(1, -1)) ** 2
            k = int(round(tmp.numel() * sparsity))
            mask1 = torch.zeros(tmp.numel(), dtype=torch.bool, device=W.device)
            if k > 0: mask1[torch.argsort(tmp.flatten())[:k]] = True   # exactly k weights (no tie overshoot)
            mask1 = mask1.reshape(tmp.shape)
        else:
            mask1 = torch.zeros_like(W1, dtype=torch.bool)
        for i in range(count):
            col = i1 + i
            w = W1[:, i]; d = Hinv1[i, i]
            q = w.clone(); q[mask1[:, i]] = 0
            if quant:
                if col % groupsize == 0:
                    g = col // groupsize
                    if fixed_grid is not None:
                        scale, zero = fixed_grid[g]
                    else:
                        blk = W1[:, i:i + groupsize] * (~mask1[:, i:i + groupsize])
                        scale, zero, _ = find_params(blk, bits)
                    grid[g] = (scale, zero)
                scale, zero = grid[col // groupsize]
                q = quantize(q, scale, zero, maxq)
                q[mask1[:, i]] = 0
            Q1[:, i] = q
            err = (w - q) / d
            W1[:, i:] -= err.unsqueeze(1).matmul(Hinv1[i, i:].unsqueeze(0))
            Err1[:, i] = err
        W[:, i1:i2] = Q1
        mask[:, i1:i2] = mask1
        W[:, i2:] -= Err1.matmul(Hinv[i1:i2, i2:])
    return W, mask, grid

def compress_weight(W, H, sparsity, bits, order, groupsize=128):
    """order in {'dense','prune','quant','joint','p_then_q','q_then_p'}"""
    if order == 'dense':
        return W.clone().float(), torch.zeros_like(W, dtype=torch.bool)
    if order == 'prune':
        Wh, m, _ = obs_sweep(W, H, sparsity=sparsity, bits=16, groupsize=groupsize); return Wh, m
    if order == 'quant':
        Wh, m, _ = obs_sweep(W, H, sparsity=0.0, bits=bits, groupsize=groupsize); return Wh, m
    if order == 'joint':
        Wh, m, _ = obs_sweep(W, H, sparsity=sparsity, bits=bits, groupsize=groupsize); return Wh, m
    if order == 'p_then_q':
        Wp, m, _ = obs_sweep(W, H, sparsity=sparsity, bits=16, groupsize=groupsize)
        Wh, m2, _ = obs_sweep(Wp, H, bits=bits, fixed_mask=m, groupsize=groupsize); return Wh, m2
    if order == 'q_then_p':
        Wq, _, grid = obs_sweep(W, H, sparsity=0.0, bits=bits, groupsize=groupsize)
        Wh, m, _ = obs_sweep(Wq, H, sparsity=sparsity, bits=bits, fixed_grid=grid, groupsize=groupsize); return Wh, m
    raise ValueError(order)

def rel_recon_error(W, Wh, H):
    D = (W.float() - Wh.float())
    num = torch.trace(D @ H @ D.T); den = torch.trace(W.float() @ H @ W.float().T)
    return (num / den).item()

def max_levels_per_group(Wh, groupsize, sample_rows=64):
    rows, cols = Wh.shape
    r = torch.linspace(0, rows - 1, min(rows, sample_rows)).long()
    m = 0
    for g0 in range(0, cols, groupsize):
        blk = Wh[r, g0:g0 + groupsize]
        for row in blk:
            m = max(m, torch.unique(row).numel())
    return m

# ---------- calibration data + layer-wise sequential compression ----------
def get_layers(model):
    return model.model.layers

@torch.no_grad()
def capture_layer0_inputs(model, calib, dev):
    layers = get_layers(model)
    store = {'x': [], 'kw': None, 'pos': False}
    class Stop(Exception): pass
    def hook(mod, args, kwargs):
        if len(args) > 0:
            store['x'].append(args[0].detach()); store['pos'] = True
        else:
            store['x'].append(kwargs['hidden_states'].detach())
        if store['kw'] is None:
            store['kw'] = {k: v for k, v in kwargs.items() if k != 'hidden_states'}
        raise Stop
    h = layers[0].register_forward_pre_hook(hook, with_kwargs=True)
    for ids in calib:
        try: model(ids.to(dev), use_cache=False)
        except Stop: pass
    h.remove()
    return torch.cat(store['x'], 0), store['kw']

def layer_fwd(layer, x, kw):
    out = layer(x, **kw)
    return out[0] if isinstance(out, (tuple, list)) else out

@torch.no_grad()
def compress_model(model, calib, sparsity, bits, order, dev, groupsize=128, verbose=True, log=print):
    """Sequential layer-wise compression. Returns per-sublayer diagnostics list."""
    layers = get_layers(model)
    inps, kw = capture_layer0_inputs(model, calib, dev)
    diags = []
    for li, layer in enumerate(layers):
        lins = {n: m for n, m in layer.named_modules() if isinstance(m, nn.Linear)}
        Hs = {n: torch.zeros(m.in_features, m.in_features, device=dev) for n, m in lins.items()}
        cnt = {n: 0 for n in lins}
        def mk(n):
            def f(mod, inp, out):
                x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()
                t = x.shape[0]
                Hs[n] *= cnt[n] / (cnt[n] + t); cnt[n] += t
                x = math.sqrt(2 / cnt[n]) * x
                Hs[n] += x.T @ x
            return f
        hs = [m.register_forward_hook(mk(n)) for n, m in lins.items()]
        for j in range(inps.shape[0]):
            layer_fwd(layer, inps[j:j + 1], kw)
        for h in hs: h.remove()
        for n, m in lins.items():
            t0 = time.time()
            W = m.weight.data
            Wh, mask = compress_weight(W, Hs[n], sparsity, bits, order, groupsize)
            e = rel_recon_error(W, Wh, Hs[n])
            m.weight.data = Wh.to(W.dtype)
            Wc = m.weight.data
            mfrac = mask.float().mean().item()
            zfrac = (Wc == 0).float().mean().item()
            mask_kept = bool((Wc[mask] == 0).all().item()) if mask.any() else True
            lev = max_levels_per_group(Wc.float(), groupsize) if bits < 16 else -1
            d = dict(layer=li, name=n, shape=list(W.shape), mask_frac=mfrac, zero_frac=zfrac, mask_kept=mask_kept,
                     max_levels=lev, rel_err=e, sec=time.time() - t0)
            diags.append(d)
            if verbose:
                log(f"  L{li:02d} {n:24s} {str(tuple(W.shape)):13s} mask={mfrac:.3f} zeros={zfrac:.3f} maskKept={mask_kept!s:5s} levels/grp={lev:3d} relErr={e:.3e} {d['sec']:.1f}s")
        del Hs
        for j in range(inps.shape[0]):
            inps[j:j + 1] = layer_fwd(layer, inps[j:j + 1], kw)
        if dev == 'cuda': torch.cuda.empty_cache()
    return diags

@torch.no_grad()
def layerwise_reproduces_model(model, calib, dev):
    """Validation: running layers one-by-one with captured kwargs reproduces the model's own last-layer output."""
    layers = get_layers(model)
    ref = {}
    h = layers[-1].register_forward_hook(lambda m, i, o: ref.__setitem__('y', (o[0] if isinstance(o, (tuple, list)) else o).detach()))
    model(calib[0].to(dev), use_cache=False); h.remove()
    x, kw = capture_layer0_inputs(model, calib[:1], dev)
    for layer in layers: x = layer_fwd(layer, x, kw)
    return (x.float() - ref['y'].float()).abs().max().item(), ref['y'].float().abs().max().item()

@torch.no_grad()
def perplexity(model, enc_ids, seqlen, dev, max_windows=None):
    n = enc_ids.numel() // seqlen
    if max_windows: n = min(n, max_windows)
    nll = 0.0
    for i in range(n):
        ids = enc_ids[:, i * seqlen:(i + 1) * seqlen].to(dev)
        logits = model(ids, use_cache=False).logits.float()
        loss = nn.functional.cross_entropy(logits[0, :-1], ids[0, 1:], reduction='sum')
        nll += loss.item()
    return math.exp(nll / (n * (seqlen - 1))), n

In [ ]:
# Cell 4 — data, model loading, lm-eval wrapper, results file
import os, json, random, numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm_eval import simple_evaluate
from lm_eval.models.huggingface import HFLM

DEV = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("compute dtype:", DTYPE)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/esann2027"
else:
    OUT_DIR = "/content/esann2027"
os.makedirs(OUT_DIR, exist_ok=True)
RES_FILE = f"{OUT_DIR}/results_{MODE}.jsonl"
print("results ->", RES_FILE)

def load_results():
    if not os.path.exists(RES_FILE): return []
    return [json.loads(l) for l in open(RES_FILE) if l.strip()]
def save_result(r):
    with open(RES_FILE, "a") as f: f.write(json.dumps(r) + "\n")

c4 = load_dataset("allenai/c4", data_files={"validation": "en/c4-validation.00000-of-00008.json.gz"}, split="validation")
wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
print("C4 docs:", len(c4), "| wikitext-2 test rows:", len(wt))

def get_calib(tok, n, seqlen, seed):
    rng = random.Random(seed); out = []
    while len(out) < n:
        doc = c4[rng.randint(0, len(c4) - 1)]["text"]
        ids = tok(doc, return_tensors="pt").input_ids
        if ids.shape[1] <= seqlen: continue
        s = rng.randint(0, ids.shape[1] - seqlen - 1)
        out.append(ids[:, s:s + seqlen])
    return out

def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to(DEV).eval()
    model.config.use_cache = False
    return model, tok

def run_lmeval(lm):
    out = simple_evaluate(model=lm, tasks=TASKS, num_fewshot=0, limit=EVAL_LIMIT, bootstrap_iters=0,
                          log_samples=False, verbosity="ERROR")
    res = {}
    for t in TASKS:
        r = out["results"].get(t, {})
        res[t] = {k: v for k, v in r.items() if k.startswith("acc") and isinstance(v, (int, float))}
    return res

def main_scores(ev):
    return {t: ev[t].get(MAIN_METRIC[t], float("nan")) for t in TASKS}

In [ ]:
# Cell 5 — one configuration = compress + perplexity + zero-shot tasks (prints everything)
def run_config(model, tok, lm, dense_sd, calib, wt_ids, model_name, order, sp, bits, seed=0):
    model.load_state_dict(dense_sd)
    t0 = time.time()
    print(f"\n=== {model_name} | order={order} | sparsity={sp:.3f} | bits={bits} | seed={seed} | TCr={tcr(sp, bits):.4f} ===")
    diags = compress_model(model, calib, sp, bits, order, DEV, groupsize=GROUPSIZE, verbose=True) if order != "dense" else []
    t_comp = time.time() - t0
    ppl, nwin = perplexity(model, wt_ids, SEQLEN, DEV, PPL_WINDOWS)
    print(f"  wikitext-2 ppl = {ppl:.3f} over {nwin} windows")
    t1 = time.time(); ev = run_lmeval(lm); t_eval = time.time() - t1
    ms = main_scores(ev)
    print("  tasks:", " | ".join(f"{t}={ms[t]:.4f}" for t in TASKS))
    me = float(np.mean([d["rel_err"] for d in diags])) if diags else 0.0
    print(f"  mean layer relErr={me:.4e} | compress {t_comp:.0f}s | eval {t_eval:.0f}s")
    return dict(model=model_name, order=order, sparsity=sp, bits=bits, tcr=tcr(sp, bits), ppl=ppl, ppl_windows=nwin,
                tasks=ev, main=ms, mean_rel_err=me, diags=diags, t_compress=t_comp, t_eval=t_eval,
                n_calib=N_CALIB, seqlen=SEQLEN, eval_limit=EVAL_LIMIT, seed=seed)

def done_key(r): return (r["model"], r["order"], round(r["sparsity"], 4), r["bits"], r.get("seed", 0))

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
done = {done_key(r) for r in load_results()}
print("already done:", len(done))

In [ ]:
# Cell 6 — RUN (pilot: one small model, 6 configs, then PASS/FAIL report)
all_rows = []
for model_name in MODELS:
    model, tok = load_model(model_name)
    print(f"\n##### {model_name}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params, {len(get_layers(model))} layers")
    calib = get_calib(tok, N_CALIB, SEQLEN, SEED)
    wt_ids = tok("\n\n".join(wt["text"]), return_tensors="pt").input_ids
    print(f"calibration: {len(calib)} x {tuple(calib[0].shape)} | wikitext tokens: {wt_ids.numel()}")
    x0, kw0 = capture_layer0_inputs(model, calib[:2], DEV)
    print("captured layer-0 input:", tuple(x0.shape), "| kwargs:", {k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in kw0.items()})
    diff, scale = layerwise_reproduces_model(model, calib, DEV)
    print(f"layer-by-layer reproduction: max|diff|={diff:.3e} (activation scale {scale:.2f})")
    dense_sd = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
    lm = HFLM(pretrained=model, tokenizer=tok, batch_size=16)
    calibs = {0: calib}
    for (order, sp, bits, sd) in GRID + EXTRA_SEEDS.get(model_name, []):
        key = (model_name, order, round(sp, 4), bits, sd)
        if key in done: print("skip (done):", key); continue
        if sd not in calibs: calibs[sd] = get_calib(tok, N_CALIB, SEQLEN, sd)
        t_cfg = time.time()
        r = run_config(model, tok, lm, dense_sd, calibs[sd], wt_ids, model_name, order, sp, bits, seed=sd)
        r["repro_diff"], r["repro_scale"] = diff, scale
        save_result(r); all_rows.append(r); done.add(key)
        n_done = len(done); el = time.time() - t_cfg
        print(f"  >>> saved. progress {n_done}/{n_total} configs | this config {el/60:.1f} min")
    del model, lm, dense_sd; gc.collect(); torch.cuda.empty_cache()

# ---------------- PILOT VALIDATION REPORT ----------------
if MODE == "pilot":
    R = {r["order"]: r for r in load_results() if r["model"] == MODELS[0]}
    checks = []
    def chk(name, ok, info="", level="FAIL"): checks.append((("PASS" if ok else level), name, info))
    d = R.get("dense")
    chk("all 6 configs produced a result", len(R) == 6, f"{sorted(R)}")
    if d:
        rp = d["repro_diff"] / max(d["repro_scale"], 1e-6)
        chk("layer-by-layer pass reproduces the model", rp < 2e-2, f"rel diff {rp:.2e}")
        chk("dense perplexity finite and plausible", math.isfinite(d["ppl"]) and d["ppl"] < 100, f"ppl {d['ppl']:.2f}")
        for t in TASKS:
            v = d["main"][t]
            chk(f"lm-eval {t} returned a valid score", isinstance(v, float) and 0 <= v <= 1, f"{v}")
            chk(f"dense {t} above chance", v > CHANCE[t], f"{v:.3f} vs chance {CHANCE[t]}", level="WARN")
    for o in ["prune", "quant", "joint", "p_then_q", "q_then_p"]:
        r = R.get(o)
        if not r: continue
        ds = r["diags"]
        n_lin = len(ds)
        chk(f"{o}: every linear layer compressed", n_lin > 0 and n_lin % len(ds and set(x['layer'] for x in ds)) == 0, f"{n_lin} sublayers")
        if r["sparsity"] > 0:
            mf = [x["mask_frac"] for x in ds]
            chk(f"{o}: mask sparsity = target", all(abs(m - r["sparsity"]) < 0.01 for m in mf), f"min {min(mf):.4f} max {max(mf):.4f} target {r['sparsity']}")
            chk(f"{o}: pruned weights still zero after quantization/cast", all(x["mask_kept"] for x in ds))
        if r["bits"] < 16:
            lv = max(x["max_levels"] for x in ds)
            chk(f"{o}: <= 2^bits levels per group", lv <= 2 ** r["bits"], f"max levels {lv} (limit {2 ** r['bits']})")
        chk(f"{o}: reconstruction error > 0 and finite", all(math.isfinite(x["rel_err"]) and x["rel_err"] > 0 for x in ds),
            f"mean {r['mean_rel_err']:.3e}")
        chk(f"{o}: perplexity finite", math.isfinite(r["ppl"]), f"{r['ppl']:.2f}")
        if d: chk(f"{o}: perplexity >= dense (sanity)", r["ppl"] >= d["ppl"] * 0.99, f"{r['ppl']:.2f} vs {d['ppl']:.2f}", level="WARN")
    if all(k in R for k in ["prune", "quant", "joint"]):
        ep, eq, ej = R["prune"]["mean_rel_err"], R["quant"]["mean_rel_err"], R["joint"]["mean_rel_err"]
        chk("joint error >= max(prune, quant) error", ej >= max(ep, eq) * 0.95, f"p={ep:.3e} q={eq:.3e} j={ej:.3e} (p+q={ep+eq:.3e})", level="WARN")
        print(f"\nInteraction preview (layer error): joint={ej:.3e} vs additive p+q={ep+eq:.3e}")
        if d:
            sr = lambda r: sum(r["main"].values()) / sum(d["main"].values())
            print(f"Interaction preview (task retention): Sr_j={sr(R['joint']):.4f} vs Sr_p*Sr_q={sr(R['prune'])*sr(R['quant']):.4f}")
    chk("tcr(0.25, 4) = 0.8125", abs(tcr(0.25, 4) - 0.8125) < 1e-9)
    print("\n================ PILOT REPORT ================")
    for s, n, i in checks: print(f"[{s}] {n}  {i}")
    nf = sum(s == "FAIL" for s, _, _ in checks)
    tc = np.mean([r["t_compress"] for r in R.values() if r["order"] != "dense"]); te = np.mean([r["t_eval"] for r in R.values()])
    print(f"\nmean time per config: compress {tc:.0f}s + eval {te:.0f}s (pilot sizes)")
    print("PILOT OK -> send me this output, then set MODE='full'" if nf == 0 else f"PILOT HAS {nf} FAILURE(S) -> send me this output")

In [ ]:
# Cell 7 — summary table (works for pilot and full results)
import pandas as pd
rows = load_results()
df = pd.DataFrame([{k: r.get(k, 0) for k in ["model", "order", "sparsity", "bits", "seed", "tcr", "ppl", "mean_rel_err"]} | r["main"] for r in rows])
def sr(g):
    d = g[g.order == "dense"].iloc[0]
    g = g.copy()
    g["Sr_tasks"] = g[TASKS].sum(1) / d[TASKS].sum()
    g["Sr_ppl"] = d.ppl / g.ppl
    return g
df = pd.concat([sr(df[df.model == m]) for m in df.model.unique()])
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 300)
print(df.round(4).to_string(index=False))